In [ ]:
!pip install -U -q accelerate transformers
!pip install -U -q bitsandbytes
!pip install -q pymupdf
!pip install -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 62.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import urllib.request
import urllib.parse
from collections import defaultdict, deque
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig)
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize
import fitz
import time
import random

In [ ]:
from google.colab import drive
# Connecting to Google Drive
drive.mount('/content/drive')

# Define directory on drive so data stays during runtime disconnects
DRIVE_DIR = "/content/drive/MyDrive/apt_extraction"
os.makedirs(DRIVE_DIR, exist_ok=True)

# Setting file paths for saving progress and raw results
PROGRESS_PATH = os.path.join(DRIVE_DIR, "extraction_progress.json")
CANDIDATES_PATH = os.path.join(DRIVE_DIR, "raw_llm_candidates.json")

print(f"Progress will be persisted to: {DRIVE_DIR}")
print(f"  progress   -> {PROGRESS_PATH}")
print(f"  candidates -> {CANDIDATES_PATH}")


Mounted at /content/drive
Progress will be persisted to: /content/drive/MyDrive/apt_extraction
  progress   -> /content/drive/MyDrive/apt_extraction/extraction_progress.json
  candidates -> /content/drive/MyDrive/apt_extraction/raw_llm_candidates.json


In [ ]:
# Mapping different verb forms to standard action names
RELATION_NORMALIZATION = {
    "used": "uses",
    "using": "uses",
    "utilizes": "uses",
    "leverages": "uses",
    "leveraged": "uses",
    "target": "targets",
    "targeted": "targets",
    "targeting": "targets",
    "exploit": "exploits",
    "exploited": "exploits",
    "download": "downloads",
    "downloaded": "downloads",
    "deliver": "delivers",
    "delivered": "delivers",
    "communicates with": "communicates-with",
    "attributed to": "attributed-to",
    "variant of": "variant-of",
    "beacons to": "beacons-to",
    "consists of": "consists-of",
    "exfiltrates to": "exfiltrates-to",
    "originates from": "originates-from",
    "based on": "based-on",
    "duplicate of": "duplicate-of",
    "related to": "related-to",
    "located at": "located-at",
}

# List of allowed actions/relationships
ALLOWED_RELATIONS = [
    "uses","exploits", "targets", "attributed-to", "downloads",
    "authored-by", "variant-of", "communicates-with", "delivers",
    "beacons-to", "consists-of", "hosts", "impersonates",
    "exfiltrates-to", "drops", "controls", "compromises",
    "originates-from", "owns", "indicates", "based-on",
    "duplicate-of", "related-to", "located-at",
]


# List of allowed item categories
ALLOWED_ENTITIES = sorted([
    "threat-actor", "malware", "tools", "SOFTWARE", "vulnerability",
    "identity", "location", "url", "IPV4", "Infrastucture",
    "attack-pattern", "campaign", "FILEPATH", "REGISTRYKEY",
    "hash", "EMAIL", "TIME"
])

# Creating a lookup dictionary mapping lowercase names to correct cases
ENTITY_MAP = {e.lower(): e for e in ALLOWED_ENTITIES}

# Mapping aliases/ alternate names to the main categories
EXTRA_ENTITY_ALIASES = {
    "tool": "tools",
    "threat actor": "threat-actor",
    "attack pattern": "attack-pattern",
    "infra": "Infrastucture",
    "infrastructure": "Infrastucture",
    "file path": "FILEPATH",
    "file-path": "FILEPATH",
    "registry key": "REGISTRYKEY",
    "registry-key": "REGISTRYKEY",
    "ip": "IPV4",
    "ip address": "IPV4",
}

# Adding the alternate names to the main lookup dictionary
ENTITY_MAP.update(EXTRA_ENTITY_ALIASES)


# # Keywords used to identify cyber threat intelligence text
# CTI_KEYWORDS = {
#     "malware", "apt", "actor", "exploit", "cve", "attack", "campaign",
#     "backdoor", "trojan", "ransomware", "phishing", "c2", "payload",
#     "vulnerability", "deploy", "download", "beacon", "exfiltrate",
#     "spear", "lateral", "persistence", "credential", "privilege",
# }

# def is_cti_relevant(sentence: str) -> bool:
#   # Check if a sentence has at least two security keywords
#     lower = sentence.lower()
#     return sum(1 for kw in CTI_KEYWORDS if kw in lower) >= 2

In [ ]:
# Fetching the list of all files in the GitHub repository
def get_repo(owner="blackorbird", repo="APT_REPORT",
                  branch="master") -> dict:
    url = (f"https://api.github.com/repos/{owner}/{repo}"
           f"/git/trees/{branch}?recursive=1")
    with urllib.request.urlopen(url) as resp:
        return json.load(resp)

# function to clean text for easier matching (lowercase, no spaces/dashes)
def _norm(s: str) -> str:
    return s.lower().replace(" ", "").replace("-", "").replace("_", "")

# Selecting suitable files from the repository based on size, type and threat actor
def select_files(tree: dict, max_files: int = 100, max_file_size_mb: float = 8,exclude_paths: set = None) -> list:
    exclude_paths = exclude_paths or set()
    # List of target threat actors to include
    THREAT_ACTORS = {"apt28", "apt29", "apt32", "apt33", "apt34", "apt38", "apt41", "kimsuky", "lazarus", "sandworm", "gamaredon", "turla", "fin7", "fin8", "darkside", "revil", "conti", "muddywater",}
    # Folders to exclude
    EXCLUDE = {"summary", "cybercrime", "aisecurity", "ot","international strategic"}

# Filter for document files that have not been processed yet
    blobs = [t for t in tree["tree"]
        if t["type"] == "blob"
        and t["path"].lower().endswith((".pdf", ".txt", ".md"))
        and t["path"] not in exclude_paths]

# Grouping eligible files by their top-level folder
    by_folder: dict = defaultdict(list)
    for f in blobs:
        folder = f["path"].split("/")[0]
        if folder.lower() in EXCLUDE:
            continue
        if f["path"].split("/")[-1].lower() in ("readme.md", "readme.txt"):
            continue
        size_mb = f.get("size", 0) / 1e6
        if size_mb > max_file_size_mb:
            continue
        if _norm(folder) in {_norm(a) for a in THREAT_ACTORS}:
            by_folder[folder].append(f)

    # Sorting files by size for each folder
    queues = {k: deque(sorted(v, key=lambda x: x.get("size", 0))) for k, v in by_folder.items()}
    # Picking files in a round-robin order to balance  across groups
    order = deque(queues.keys())
    selected = []
    # Selecting files until the maximum number is reached or no files remain
    while order and len(selected) < max_files:
        folder = order.popleft()
        q = queues[folder]
        if q:
            selected.append(q.popleft())
            if q:
                order.append(folder)

    # Getting the names of the APT groups represented in the selected files
    groups = sorted({f["path"].split("/")[0] for f in selected})
    # Printing the number of selected files and APT groups
    print(f"Selected {len(selected)} files across {len(groups)} APT groups:")
    print(" ", groups)
    return selected

# Downloading selected files from GitHub to local folder
def download_files(selected: list, out_dir: str = "apt_subset", owner: str = "blackorbird", repo: str = "APT_REPORT", branch: str = "master") -> list:
    os.makedirs(out_dir, exist_ok=True)  # Creating the output folder if it does not already exist
    downloaded = []
     # Downloading each selected file from the repository
    for i, f in enumerate(selected, 1):
        path = f["path"]
        url = (f"https://raw.githubusercontent.com/{owner}/{repo}"
               f"/{branch}/" + urllib.parse.quote(path))
       # Saving with flattened file names to avoid nested folder issues
        local = os.path.join(out_dir, path.replace("/", "__"))
        try:
            urllib.request.urlretrieve(url, local)
            downloaded.append(local)
        except Exception as e:
            print(f"  skipped {path}: {e}")
        # Printing progress every 10 downloads or on the last file
        if i % 10 == 0 or i == len(selected):
            print(f"  downloaded {i}/{len(selected)}")
    return downloaded

print("Fetching files from GitHub")
tree = get_repo()
selected = select_files(tree, max_files=2500)
report_files = download_files(selected)
print(f" {len(report_files)} files downloaded.")


Fetching files from GitHub
Selected 96 files across 10 APT groups:
  ['APT28', 'APT29', 'APT34', 'APT41', 'Gamaredon', 'Sandworm', 'Turla', 'kimsuky', 'lazarus', 'muddywater']
  downloaded 10/96
  downloaded 20/96
  downloaded 30/96
  downloaded 40/96
  downloaded 50/96
  downloaded 60/96
  downloaded 70/96
  downloaded 80/96
  downloaded 90/96
  downloaded 96/96
 96 files downloaded.


In [ ]:
# Extracting text from PDF files up to max_pages or read text files directly
def extract_text(path: str, max_pages: int = 12) -> str:
    if path.lower().endswith(".pdf"):
        try:
            doc = fitz.open(path)
            # Read only up to the page limit
            pages = doc[:max_pages] if len(doc) > max_pages else doc
            return "\n".join(page.get_text() for page in pages)
        except Exception as e:
            print(f"  PDF skipped ({path}): {e}")
            return ""
    # Open non-PDF text files
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        return fh.read()


# Read files and collect useful sentences grouped by file path
def build_sentence_pool(files: list, target_total: int = 5000) -> dict:
    pool: dict = {}
    total = 0
    for path in files:
        if total >= target_total:
            break
        text = extract_text(path)
        if not text.strip():
            continue
        # Cleaning up extra spaces and line breaks
        cleaned = re.sub(r"\s+", " ", text)

        seen = {} # Removing duplicate sentences within the same file while keeping order
        for sent in sent_tokenize(cleaned):
            sent = sent.strip()
            # Keeping the sentence if length is good and content is relevant
            if 40 < len(sent) < 600:
                seen.setdefault(sent, None)
                # Stopping if we reach our target count
                if total + len(seen) >= target_total:
                    break

        # Saving selected sentences for this file
        if seen:
            pool[path] = list(seen.keys())
            total += len(seen)

    n_sentences = sum(len(v) for v in pool.values())
    print(f"\nSentence pool: {n_sentences} CTI-relevant sentences "
          f"across {len(pool)} documents (from {len(files)} files scanned)")
    return pool

# Generating the sentence pool with up to 5,000 sentences
sentence_pool = build_sentence_pool(report_files, target_total=5000)



Sentence pool: 5000 CTI-relevant sentences across 61 documents (from 96 files scanned)


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

# Using the API key stored in Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
# Initializing the Groq client with the API key
client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialized")

# Define the qwen LLM model to use for candidate extraction
CANDIDATE_MODEL = "qwen/qwen3.6-27b"

Groq client initialized


In [ ]:
import re as _re
# Building trigger words for each relation from allowed relations and normalization mappings
RULES = { relation: {"triggers": [relation.replace("-", " ")]} for relation in ALLOWED_RELATIONS }
# Add word variations and tenses as extra triggers
for surface_form, canonical in RELATION_NORMALIZATION.items():
    RULES.setdefault(canonical, {"triggers": []})
    RULES[canonical]["triggers"].append(surface_form)


# Checking if a sentence contains key triggers before sending it to the LLM
def should_call_llm(sentence: str) -> bool:
    text = sentence.lower()

    # Return True if any relation trigger phrase is found
    for relation in RULES.values():
        for trigger in relation["triggers"]:
            if trigger.lower() in text:
                return True

    return False


In [ ]:
# function where we Build a prompt that instructs the LLM to extract CTI triples in JSON format
def build_prompt(sentences: list) -> str:
  # Format the lists of allowed relations and entities into comma-separated strings
    relations_str = ", ".join(ALLOWED_RELATIONS)
    entities_str  = ", ".join(ALLOWED_ENTITIES)
    # Number each sentence for the prompt layout
    numbered = "\n".join(f'{i+1}. "{s}"' for i, s in enumerate(sentences))

    content = f"""You are a Cyber Threat Intelligence extraction engine.

Allowed relation types : {relations_str}
Allowed entity types   : {entities_str}

You will be given {len(sentences)} numbered sentences.
For EACH sentence extract relation triples. Output ONLY a single valid JSON array
with exactly {len(sentences)} elements (one per sentence, in order).
Each element is an array of triples or [] if none apply.

For every triple, copy the head and tail entity mentions EXACTLY as they
appear in the sentence (same casing, same wording - these will be located
in the original text programmatically, so they must match verbatim).

Triple format:
{{"head":"exact entity text","head_type":"...","relation":"...","tail":"exact entity text","tail_type":"..."}}

Do NOT output any text outside the JSON array.
NOTE: spell Infrastructure as "Infrastucture".

EXAMPLE for 2 sentences:
[
  [{{"head":"Cobalt Strike","head_type":"tools","relation":"beacons-to","tail":"1.2.3.4","tail_type":"IPV4"}}],
  []
]

Sentences:
{numbered}
"""
    return content


In [ ]:
import threading
import time
import random

MAX_RETRIES = 6   # Maximum number of retry attempts
INITIAL_DELAY = 0.5   # Base delay for exponential backoff (seconds)
MAX_DELAY = 30   # Maximum wait delay (seconds)
MIN_PACING = 4.0   # Minimum gap between two requests
DEFAULT_MAX_TOKENS = 600

# Manage request timing and slow down if rate limits are hit
class RateLimiter:
    def __init__(self, min_pacing: float = MIN_PACING):
        self.min_pacing = min_pacing
        self.current_pacing = min_pacing
        self._last_call = 0.0
        self._lock = threading.Lock()

    # Wait if needed to maintain the minimum time gap between API calls
    def wait(self):
        with self._lock:
            now = time.monotonic()
            remaining = self.current_pacing - (now - self._last_call)
            if remaining > 0:
                time.sleep(remaining)
            self._last_call = time.monotonic()

    # Increase the delay between calls when hitting rate limits
    def penalize(self):
        with self._lock:
            self.current_pacing = min(self.current_pacing * 1.5, 5.0)

    # Gradually reduce delay back to normal after successful calls
    def relax(self):
        with self._lock:
            self.current_pacing = max(self.min_pacing, self.current_pacing * 0.95)

# Global rate limiter instance
rate_limiter = RateLimiter()

# Send prompt to Groq API with automatic retries and rate limiting handling
def call_groq(prompt, max_tokens: int = 600):
    # Try sending the request until it succeeds or retries are exhausted
    for attempt in range(MAX_RETRIES):
        rate_limiter.wait()
        try:
            response = client.chat.completions.create( model=CANDIDATE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
                reasoning_effort="none",
            )

            # Reduce the delay after a successful request
            rate_limiter.relax()
            return response

        except Exception as e:
            error = str(e)

            # Stop immediately if daily token quota is reached
            if "TPD" in error or "tokens per day" in error.lower():
                print("\n" + "=" * 60)
                print("Groq daily token limit reached. Stopping.")
                print("=" * 60)
                return "DAILY_LIMIT_EXCEEDED"

            # Check if error is retryable (rate limit or temporary server error)
            retryable = any(code in error for code in ["429", "500", "502", "503", "504"])
            if not retryable:
                raise

            # Stop trying if max retries are reached
            if attempt == MAX_RETRIES - 1:
                print(f"Giving up after {MAX_RETRIES} retries: {error[:200]}")
                return None

            # Wait before retrying using exponential backoff with jitter
            rate_limiter.penalize()
            wait = min(INITIAL_DELAY * (2 ** attempt), MAX_DELAY)
            wait += random.uniform(0, 0.5)
            status = "429" if "429" in error else "server error"
            print(f"{status} received. Retry {attempt+1}/{MAX_RETRIES}, " f"waiting {wait:.1f}s")
            time.sleep(wait)

    # Return None if the request could not be completed
    return None

In [ ]:
import ast
# Find and extract the first JSON array from a string using matching brackets
def extract_first_json_array(text: str):
    if not isinstance(text, str):
        return None
      # Find the opening bracket of the array
    start = text.find('[')
    if start == -1:
        return None
      # Track nested brackets to find the matching closing bracket
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == '[':
            depth += 1
        elif c == ']':
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None


# Parse JSON array text with fallback fixes for common formatting issues
def parse_json_array_flexible(text: str):
    if not isinstance(text, str):
        raise ValueError("Non-string model output")

   # Extract the array text
    arr_txt = extract_first_json_array(text)
    if arr_txt is None:
        # no top-level array found
        raise ValueError("No JSON array found in output")


    # Trying standard JSON parsing first
    try:
        return json.loads(arr_txt)
    except Exception as e_json:
        # Fallback parsing for single quotes or extra trailing commas
        safe = arr_txt
        # Converting single quotes to double quotes if no double quotes exist
        if "'" in safe and '"' not in safe:
            safe = safe.replace("'", '"')
        # removing common trailing commas inside arrays
        safe = safe.replace(",]", "]").replace(",}", "}")
        try:
            parsed = ast.literal_eval(safe)
            return parsed
        except Exception as e_ast:
            raise RuntimeError(f"json.loads error: {e_json}; ast.literal_eval error: {e_ast}")



In [ ]:
# function to remove model thinking tags (<think> and </think>)
def strip_thinking_block(text: str) -> str:
    if not isinstance(text, str):
        return text
    think_end = text.find("</think>")
    if think_end != -1:
        return text[think_end + len("</think>"):].strip()
    return text

In [ ]:
import re as _re3

# function to Extract entity texts and build clean sentence from entity-marked text
def extract_marked_entities(marked_text: str):
  # Find text inside [E1] and [E2] tags
    e1 = _re3.search(r"\[E1\]\s*(.*?)\s*\[/E1\]", marked_text)
    e2 = _re3.search(r"\[E2\]\s*(.*?)\s*\[/E2\]", marked_text)
    if not e1 or not e2:
        return None
    head = e1.group(1).strip()
    tail = e2.group(1).strip()
    if not head or not tail:
        return None
    # Strip markers to recover the original sentence
    plain = marked_text.replace(e1.group(0), head).replace(e2.group(0), tail)
    plain = _re3.sub(r"\s+", " ", plain).strip()
    return head, tail, plain


# Checking if marked text contains valid and single [E1] and [E2] tag pairs
def is_valid_marked_text(marked_text: str) -> bool:
    if not isinstance(marked_text, str):
        return False
    # Checking each tag appears exactly once
    if marked_text.count("[E1]") != 1 or marked_text.count("[/E1]") != 1:
        return False
    if marked_text.count("[E2]") != 1 or marked_text.count("[/E2]") != 1:
        return False
    return extract_marked_entities(marked_text) is not None


In [ ]:
import re as _re2

# Finding the start and end character positions of an entity inside a sentence
def find_entity_span(sentence: str, entity: str):
    if not entity or not sentence:
        return None

    entity = entity.strip()
    if not entity:
        return None

    # Trying exact case-insensitive search
    match = _re2.search(_re2.escape(entity), sentence, flags=_re2.IGNORECASE)
    if match:
        return match.start(), match.end()

    # Trying matching with  whitespace
    loose_pattern = _re2.escape(entity)
    loose_pattern = _re2.sub(r"\\ ", r"\\s+", loose_pattern)
    match = _re2.search(loose_pattern, sentence, flags=_re2.IGNORECASE)
    if match:
        return match.start(), match.end()

    return None


# Checking if two character spans overlap with each other
def spans_overlap(a_start, a_end, b_start, b_end) -> bool:
    return a_start < b_end and b_start < a_end


# Inserting entity tags ([E1], [/E1], [E2], [/E2]) into text from right to lef
def insert_entity_markers(text, head_start, head_end, tail_start, tail_end):
    # Sorting spans right-to-left so tag insertions don't alter earlier offset
    spans = sorted([(head_start, head_end, "[E1]", "[/E1]"), (tail_start, tail_end, "[E2]", "[/E2]")],  key=lambda s: -s[0])  # right to left so inserted tags don't shift earlier offsets
    out = text
    # Inserting tags around entities
    for start, end, open_tag, close_tag in spans:
        out = out[:end] + f" {close_tag}" + out[end:]
        out = out[:start] + f"{open_tag} " + out[start:]
    return out


In [ ]:
# fucntion to parse a JSON batch response from Groq and extract valid relation triples
def parse_llm_batch_response(batch_sentences: list, gen_text: str) -> list:
    gen_text = strip_thinking_block(gen_text)
    if not gen_text:
        return []

    # Finding the boundaries of the JSON array
    start_json = gen_text.find("[")
    end_json = gen_text.rfind("]")
    if start_json == -1 or end_json == -1:
        print(f"  No JSON array in response. Snippet: {gen_text[:200]}")
        return []

    # Trying to parse the JSON response
    try:
        batch_results = json.loads(gen_text[start_json:end_json + 1])
    except Exception as e:
        print(f"  JSON parse error: {e}")
        return []

    # Checking that the response contains a list of results
    if not isinstance(batch_results, list):
        return []
    if len(batch_results) != len(batch_sentences):
        print(f"  Warning: model returned {len(batch_results)} elements "
              f"for {len(batch_sentences)} sentences")

    rows = []
    dropped_not_found = 0
    dropped_overlap = 0

    # Validating and structuring each extracted triple
    for sentence, triples in zip(batch_sentences, batch_results):
        if not isinstance(triples, list):
            continue
        for triple in triples:
            if not isinstance(triple, dict):
                continue
            required = ["head", "head_type", "relation", "tail", "tail_type"]
            if not all(k in triple for k in required):
                continue

            head = str(triple["head"]).strip()
            tail = str(triple["tail"]).strip()
            if not head or not tail:
                continue

            # Standardizing the relation name and check against allowed relations
            relation = str(triple["relation"]).lower().strip()
            relation = RELATION_NORMALIZATION.get(relation, relation)
            if relation not in ALLOWED_RELATIONS:
                continue

            # Checking that the entity types are valid
            head_type = str(triple["head_type"]).lower().strip()
            tail_type = str(triple["tail_type"]).lower().strip()
            if head_type not in ENTITY_MAP or tail_type not in ENTITY_MAP:
                continue

            # Locating the entity spans in the original sentence
            head_span = find_entity_span(sentence, head)
            tail_span = find_entity_span(sentence, tail)
            if head_span is None or tail_span is None:
                dropped_not_found += 1
                continue

            # Checking that the two entity spans do not overlap
            head_start, head_end = head_span
            tail_start, tail_end = tail_span
            if spans_overlap(head_start, head_end, tail_start, tail_end):
                dropped_overlap += 1
                continue

            # Inserting entity markers into the sentence
            marked_text = insert_entity_markers(sentence, head_start, head_end, tail_start, tail_end)

            # Storing the validated relation triple
            rows.append({ "marked_text": marked_text,
                "head_type": ENTITY_MAP[head_type],
                "tail_type": ENTITY_MAP[tail_type],
                "relation": relation,
                "head": head,
                "tail": tail,
                "sentence": sentence,
            })

    # Printing triples that were removed during span matching
    if dropped_not_found or dropped_overlap:
        print(f"  Span-matching drops not found: {dropped_not_found}, overlap: {dropped_overlap}")

    return rows


# Sending a batch of sentences to the LLM for relation extraction
def extract_batch_triples(batch_sentences: list, max_tokens_per_sentence: int = 250) -> tuple:
    prompt = build_prompt(batch_sentences)
    max_tokens = max_tokens_per_sentence * len(batch_sentences)
    response = call_groq(prompt, max_tokens=max_tokens)

    # Handling daily token limit errors
    if response == "DAILY_LIMIT_EXCEEDED":
        return "daily_limit", []
    if response is None:
        return "error", []

    # Extracting the generated text from the API response
    try:
        gen_text = (response.choices[0].message.content or "").strip()
    except Exception:
        return "error", []

    # Parse and validate the extracted triples
    return "ok", parse_llm_batch_response(batch_sentences, gen_text)


# Loading the extraction progress from a JSON file
def load_progress(progress_path: str = "extraction_progress.json") -> dict:
    if os.path.exists(progress_path):
        with open(progress_path, "r") as fh:
            return json.load(fh)
    return {"processed_files": [], "in_progress": None}


# Saving the extraction progress to a JSON file
def save_progress(progress: dict, progress_path: str = "extraction_progress.json"):
    with open(progress_path, "w") as fh:
        json.dump(progress, fh, indent=2)


# Saving extracted rows to disk and remove duplicate entries
def append_candidates(new_rows: list, output_path: str = "raw_llm_candidates.json") -> pd.DataFrame:
    new_df = pd.DataFrame(new_rows)
    if os.path.exists(output_path):
        existing_df = pd.read_json(output_path)
        combined = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined = new_df

    # Removing duplicate relation examples
    if not combined.empty:
        combined = combined.drop_duplicates(subset=["marked_text", "relation"], keep="first")

    # Saving the combined results to JSON
    combined.to_json(output_path, orient="records", indent=2)
    return combined


# Processing files one by one with support for pausing and resuming
def run_extraction_session(
    sentence_pool: dict,
    target_new_files: int = 100,
    batch_size: int = 8,
    max_tokens_per_sentence: int = 700,
    progress_path: str = "extraction_progress.json",
    output_path: str = "raw_llm_candidates.json",
) -> pd.DataFrame:

    # Resuming a partially processed file if available
    progress = load_progress(progress_path)
    processed = set(progress["processed_files"])

    queue = []
    resume_idx = {}
    if progress["in_progress"]:
        f = progress["in_progress"]["file"]
        if f in sentence_pool:
            queue.append(f)
            resume_idx[f] = progress["in_progress"]["next_sentence_idx"]

    # Adding new unprocessed files to the queue
    new_files = [f for f in sentence_pool if f not in processed and f not in resume_idx]
    queue.extend(new_files[:target_new_files])

    print(f"{len(queue)} file(s) queued "
          f"({'1 resumed + ' if resume_idx else ''}"
          f"{len(queue) - len(resume_idx)} new), "
          f"{len(processed)} already fully processed in earlier sessions.")

    session_rows = []

    # Processing each queued file
    for file_path in queue:
        sentences = sentence_pool[file_path]
        start_idx = resume_idx.get(file_path, 0)
        print(f"\n[{file_path}] {len(sentences) - start_idx} sentence(s) to go"
              + (f" (resuming from {start_idx})" if start_idx else ""))

        idx = start_idx

        # Processing the sentences in batches
        while idx < len(sentences):
            batch = sentences[idx: idx + batch_size]
            status, rows = extract_batch_triples(batch, max_tokens_per_sentence=max_tokens_per_sentence)

            # Saving progress and pause if the daily token limit is reached
            if status == "daily_limit":
                progress["in_progress"] = {"file": file_path, "next_sentence_idx": idx}
                save_progress(progress, progress_path)
                combined = append_candidates(session_rows, output_path)
                print(f"\nDaily limit hit. '{file_path}' paused at sentence "
                      f"{idx}/{len(sentences)}, will resume there next session.")
                print(f"Rows saved this session: {len(session_rows)}. "
                      f"Cumulative total in {output_path}: {len(combined)}.")
                return combined

            # Adding extracted rows and update the sentence index
            session_rows.extend({**row, "source_file": file_path} for row in rows)
            idx += len(batch)
            print(f"  {idx}/{len(sentences)} sentences | "
                  f"pacing={rate_limiter.current_pacing:.2f}s | "
                  f"triples so far this session={len(session_rows)}")

        # Marking the file as fully processed
        processed.add(file_path)
        progress["processed_files"] = sorted(processed)
        if progress["in_progress"] and progress["in_progress"]["file"] == file_path:
            progress["in_progress"] = None
        save_progress(progress, progress_path)

    # Saving the final extracted results
    combined = append_candidates(session_rows, output_path)
    print(f"\nSession complete. {len(queue)} file(s) processed this session "
          f"({len(processed)} total done across all sessions).")
    print(f"New triples this session: {len(session_rows)}. "
          f"Cumulative total in {output_path}: {len(combined)}.")
    return combined

In [ ]:
# Print sentence pool summary and configured model
print(f"Sentence pool: {sum(len(v) for v in sentence_pool.values())} sentences "
      f"across {len(sentence_pool)} documents. Model={CANDIDATE_MODEL}\n")

# execution start time
start_time = time.time()

# batch extraction session
raw_candidates_df = run_extraction_session( sentence_pool,
    target_new_files=100,
    batch_size=8,
    max_tokens_per_sentence=250,
    progress_path=PROGRESS_PATH,
    output_path=CANDIDATES_PATH,
)

elapsed = time.time() - start_time
print(f"\nSession runtime: {elapsed:.1f} seconds.")
print(f"Total candidate triples saved so far (all sessions): {len(raw_candidates_df)}")


Sentence pool: 5000 sentences across 61 documents. Model=qwen/qwen3.6-27b

10 file(s) queued (1 resumed + 9 new), 51 already fully processed in earlier sessions.

[apt_subset/Sandworm__russia-nexus-uac-0113-emulating-telecommunication-providers-in-ukraine.pdf] 122 sentence(s) to go (resuming from 48)
  56/170 sentences | pacing=4.00s | triples so far this session=0
  64/170 sentences | pacing=4.00s | triples so far this session=0
  Span-matching drops -- not found: 3, overlap: 1
  72/170 sentences | pacing=4.00s | triples so far this session=5
  80/170 sentences | pacing=4.00s | triples so far this session=5
  Span-matching drops -- not found: 3, overlap: 0
  88/170 sentences | pacing=4.00s | triples so far this session=7
  Span-matching drops -- not found: 1, overlap: 0
  96/170 sentences | pacing=4.00s | triples so far this session=8
  104/170 sentences | pacing=4.00s | triples so far this session=14
  112/170 sentences | pacing=4.00s | triples so far this session=20
  120/170 senten

In [ ]:
# Printing a sample of the extracted candidate triples
print("Sample candidates")
# Check if any candidate triples were extracted
if not raw_candidates_df.empty:
    # Selecting the main columns to display
    cols = ["marked_text", "head_type", "relation", "tail_type"]
    # Printing the first 20 candidate triples
    print(raw_candidates_df[cols].head(20).to_string(index=False))

    # Print the distribution of relation types
    print("\nRelation distribution:")
    print(raw_candidates_df["relation"].value_counts().to_string())

    # Printing the distribution of head entity types
    print("\nHead type distribution:")
    print(raw_candidates_df["head_type"].value_counts().to_string())
    # Printing the number of source documents and top documents by triple count
    if "source_file" in raw_candidates_df.columns:
        print(f"\nTriples span {raw_candidates_df['source_file'].nunique()} distinct source documents.")
        print("Triples per document (top 10):")
        print(raw_candidates_df["source_file"].value_counts().head(10).to_string())

else:

    # Printing a message if no candidates were extracted
    print("No candidates extracted yet.")

Sample candidates
                                                                                                                                                                                                                                                                                                                                                                       marked_text     head_type          relation     tail_type
                                                                                                                             related report : https://www.securityartwork.es/2019/04/04/ukraine-election-2019-polls-maldoc-analysis/ [E1] Ukraine_election_2019_polls.doc [/E1] [E2] 8a35b6ecdf43f42dbf1e77235d6017faa70d9c68930bdc891d984a89d895c1e7 [/E2] C2：functiondiscovery[.      FILEPATH         indicates          hash
                                                                                                                             related report : https:

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/apt_extraction/raw_llm_candidates.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>